**Setup**

In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [7]:
M_xanthus = read_sbml_model("/home/mickael/github/M_xanthus-E_coli-Predation/M_xanthus_model_V3_hdca.xml")
M_xanthus

Name,myxo_model
Memory address,7d2406548a40
Number of metabolites,1224
Number of reactions,1339
Number of genes,1201
Number of groups,0
Objective expression,1.0*OF_BIOMASS - 1.0*OF_BIOMASS_reverse_80d2e
Compartments,"c, e"


In [8]:
# np.random.seed(1) # set a seed to have similar results if repeated

**Genetic Algorithm**

Create first individual

In [9]:
Exchange_list = []
for i in M_xanthus.exchanges._dict:
    Exchange_list.append(i) # get the list of the exchange reaction

n = 500
individual = [] # one individual = dict of EX reaction and their lower bound

for i in range(n):
    dico_temp = {}
    for j in Exchange_list:
        dico_temp[j] = np.random.randint(-1000,0) # create all the individuals with random lower bound (importation)
    individual.append(dico_temp)

run the algorithm

In [10]:
generation = 0 # the current generation
final = 50 # the last generation wanted

with tqdm(total=final) as pbar:
    while generation <= final:
        fitness = [] # store all the objective value
        for i in individual:
            for j in i:
                M_xanthus.reactions.get_by_id(j).lower_bound = i[j]
            FBA = M_xanthus.optimize()
            fitness.append(FBA.objective_value)

        sorted_list = fitness.copy() # copy the list
        sorted_list.sort(reverse=True) # sort it decreasing (best first) 

        best_index = []
        for i in range(int(30/100 * len(sorted_list))):
            best_index.append(fitness.index(sorted_list[i])) # take the index of the top 30%
            
        individual2 = []
        for b in best_index:
            individual2.append(individual[b]) # keep the top 30% in next generation

        for i in range(n-30):
            dico_temp = {}
            for j in Exchange_list:
                dico_temp[j] = individual[np.random.choice(best_index)][j] # create the rest with value randomly take from the top 30%
            
            individual2.append(dico_temp)

            mut = np.random.randint(0,100) # look for if there is a mutation
            if mut >= 95:
                if np.random.randint(1,3) == 1: # switch
                    choosen_reaction_1 = np.random.choice(Exchange_list)
                    choosen_reaction_2 = np.random.choice(Exchange_list)
                    store = individual2[i][choosen_reaction_1]

                    individual2[i][choosen_reaction_1] = individual2[i][choosen_reaction_2]
                    individual2[i][choosen_reaction_2] = store

                if np.random.randint(1,3) == 2: # change the value
                    choosen_reaction = np.random.choice(Exchange_list)
                    individual2[i][choosen_reaction] = np.random.randint(-1000, 0)
                
                if np.random.randint(1,3) == 3: # scramble
                    choosen_reaction_1 = np.random.choice(Exchange_list)
                    choosen_reaction_2 = np.random.choice(Exchange_list)
                    choosen_reaction_3 = np.random.choice(Exchange_list)
                    choosen_reaction_4 = np.random.choice(Exchange_list)
                    store = [individual2[i][choosen_reaction_1],individual2[i][choosen_reaction_2],individual2[i][choosen_reaction_3],individual2[i][choosen_reaction_4]]

                    individual2[i][choosen_reaction_1] = np.random.choice(store, replace = False)
                    individual2[i][choosen_reaction_2] = np.random.choice(store, replace = False)
                    individual2[i][choosen_reaction_3] = np.random.choice(store, replace = False)
                    individual2[i][choosen_reaction_4] = np.random.choice(store, replace = False)


        
        individual = individual2 # set the new generation
        generation += 1
        pbar.update(1)

print("Maximum = " + str(max(fitness)) + ": " + str(individual[fitness.index(max(fitness))]))
for i in range(len(individual)):
    print(str(fitness[i]) + ": " + str(individual[i])) # print the last generation

51it [16:01, 18.84s/it]                        

Maximum = 221.8813016225879: {'EX_malt_e': -70, 'EX_his_L_e': -187, 'EX_cd2_e': -997, 'EX_ferrich_e': -980, 'EX_tttnt_e': -83, 'EX_metox_e': -121, 'EX_btn_e': -210, 'EX_met_L_e': -1000, 'EX_mg2_e': -723, 'EX_ac_e': -583, 'EX_arsenb_e': -881, 'EX_cbl1_e': -133, 'EX_cgly_e': -103, 'EX_fum_e': -656, 'EX_h2s_e': -161, 'EX_so4_e': -436, 'EX_pydx_e': -74, 'EX_spmd_e': -112, 'EX_ppi_e': -818, 'EX_mnl_e': -35, 'EX_pro_L_e': -832, 'EX_glyc3p_e': -379, 'EX_orn_e': -378, 'EX_salcn_e': -663, 'EX_pi_e': -813, 'EX_acgam_e': -211, 'EX_pyr_e': -257, 'EX_etoh_e': -642, 'EX_but_e': -479, 'EX_hom_L_e': -166, 'EX_na1_e': -222, 'EX_acald_e': -802, 'EX_mobd_e': -795, 'EX_metox_R_e': -3, 'EX_arbt_e': -961, 'EX_sucr_e': -467, 'EX_galt_e': -449, 'EX_glyb_e': -834, 'EX_leu_L_e': -187, 'EX_h2o_e': -487, 'EX_so3_e': -810, 'EX_h2_e': -959, 'EX_lys_L_e': -549, 'EX_k_e': -662, 'EX_n2o_e': -809, 'EX_cu2_e': -667, 'EX_Fe3_e': -472, 'EX_fru_e': -183, 'EX_gam_e': -423, 'EX_gly_cys_L_e': -173, 'EX_alaala_e': -578, 'EX_co

Separate steps

In [11]:
# fitness = []
# for i in individual:
#     for j in i:
#         M_xanthus.reactions.get_by_id(j).lower_bound = i[j]
#     FBA = M_xanthus.optimize()
#     fitness.append(FBA.objective_value)

# print(fitness)

In [12]:
# sorted_list = fitness.copy()
# sorted_list.sort(reverse=True)

# best_index = []

# for i in range(int(30/100 * len(sorted_list))):
#     best_index.append(fitness.index(sorted_list[i]))

# print(best_index)

In [13]:
# individual2 = []
# for b in best_index:
#     individual2.append(individual[b])

# for i in range(n-30):
#     dico_temp = {}
#     for j in Exchange_list:
#         dico_temp[j] = individual[np.random.choice(best_index)][j]
#     individual2.append(dico_temp)

# individual = individual2

In [14]:
# # mutation
# mut = np.random.randint(0,100)
# if mut >= 95:
#     choosen_ind = np.random.randint(0,len(individual2))
#     if np.random.randint(1,3) == 1: # switch
#         choosen_reaction_1 = np.random.choice(Exchange_list)
#         choosen_reaction_2 = np.random.choice(Exchange_list)
#         store = individual2[choosen_ind][choosen_reaction_1]

#         individual2[choosen_ind][choosen_reaction_1] = individual2[choosen_ind][choosen_reaction_2]
#         individual2[choosen_ind][choosen_reaction_2] = store

#     if np.random.randint(1,3) == 2: # change the value
#         choosen_reaction = np.random.choice(Exchange_list)
#         individual2[choosen_ind][choosen_reaction] = np.random.randint(-1000, 0)
    
#     if np.random.randint(1,3) == 3: # scramble
#         choosen_reaction_1 = np.random.choice(Exchange_list)
#         choosen_reaction_2 = np.random.choice(Exchange_list)
#         choosen_reaction_3 = np.random.choice(Exchange_list)
#         choosen_reaction_4 = np.random.choice(Exchange_list)
#         store = [individual2[choosen_ind][choosen_reaction_1],individual2[choosen_ind][choosen_reaction_2],individual2[choosen_ind][choosen_reaction_3],individual2[choosen_ind][choosen_reaction_4]]

#         individual2[choosen_ind][choosen_reaction_1] = np.random.choice(store, replace = False)
#         individual2[choosen_ind][choosen_reaction_2] = np.random.choice(store, replace = False)
#         individual2[choosen_ind][choosen_reaction_3] = np.random.choice(store, replace = False)
#         individual2[choosen_ind][choosen_reaction_4] = np.random.choice(store, replace = False)

